In [1]:
import pandas as pd
import numpy as np
import sys
import warnings
import os

In [15]:
pd.set_option("display.max_columns", None) # Set to view all columns

In [3]:
print(sys.executable) # Check environment

C:\Users\olgan\anaconda3\envs\Test_env_1\python.exe


In [16]:
df = pd.read_csv("...data/UK_SanctionsDataRaw.csv", skiprows=1, on_bad_lines='warn', low_memory = False)
# Load data

FileNotFoundError: [Errno 2] No such file or directory: '...data/UK_SanctionsDataRaw.csv'

In [5]:
# check if Unique ID is individual AND company/entity
type_per_id = df.groupby("Unique ID")["Designation Type"].nunique()
conflicts = type_per_id[type_per_id > 1]

print(f"Unique IDs with mixed Designation Types: {len(conflicts)}")
if len(conflicts) > 0:
    print(conflicts)
    # show what those mixed records look like if they exist
    df[df["Unique ID"].isin(conflicts.index)][["Unique ID", "Designation Type", "Name 6"]].head(10)

Unique IDs with mixed Designation Types: 0


In [6]:
# check row distribution for Unique entities
rows_per_entity = df.groupby('Unique ID').size()

print(f"Total unique entities: {df['Unique ID'].nunique()}")
print(f"Total rows: {len(df)}")
print(f"Average rows per entity: {rows_per_entity.mean():.2f}")
print(f"Max rows for a single entity: {rows_per_entity.max()}")
print(f"\nDistribution:")
print(rows_per_entity.value_counts().sort_index().head(20))

# Which entity has the most rows?
most_rows = rows_per_entity.idxmax()
print(f"\nEntity with most rows: {most_rows} ({rows_per_entity.max()} rows)")

Total unique entities: 6046
Total rows: 57033
Average rows per entity: 9.43
Max rows for a single entity: 3780

Distribution:
1     2920
2     1123
3      390
4      428
5      134
6      213
7       43
8      136
9       62
10      50
11      12
12      91
13       7
14      25
15      15
16      52
17       1
18      39
19       1
20      28
Name: count, dtype: int64

Entity with most rows: INU0075 (3780 rows)


In [7]:
# Core functions - edit as applicable

# Strip Numeric Values
def numeric_proc(df):
    # strip numeric cols
    numeric_cols = ["Passport number", "National Identifier number", "Phone number", "Business registration number (s)", "IMO number"]
    for col in numeric_cols:
        if col in df:
            df[col] = df[col].str.lstrip("'").str.strip()

    # join name cols
    insert_loc = df.columns.get_loc("Name 1") # get location of column
    
    given_cols = ["Name 1", "Name 2", "Name 3", "Name 4", "Name 5"]

    # temp variables for name cols
    temp_first = df[given_cols].apply(lambda x: ' '.join(x.dropna().str.strip().str.upper()), axis=1).replace('', np.nan)
    temp_surname = df["Name 6"].str.strip().str.upper()
    temp_full = pd.concat([temp_first, temp_surname], axis=1).apply(lambda x: ' '.join(x.dropna()), axis=1).replace('', np.nan)

    # drop unnecessary cols
    df = df.drop(columns = given_cols + ["Name 6"])

    # insert name cols
    df.insert(insert_loc, "First Name", temp_first)
    df.insert(insert_loc + 1, "Surname", temp_surname)
    df.insert(insert_loc + 2, "Full Name", temp_full)
    
    return df

# Reduce Address Column Numbers
def address_proc(df):
    addr_cols = [
        "Address Line 1", "Address Line 2", "Address Line 3", 
        "Address Line 4", "Address Line 5", "Address Line 6", 
        "Address Postal Code"
    ]

# validate cols actually exist
    valid_cols = [col for col in addr_cols if col in df.columns]
    if  not valid_cols:
        print("Columns do not exist")
        return df

    insert_loc = df.columns.get_loc("Address Line 1")

    temp_full_address = df[valid_cols].apply(
        lambda row: ' '.join([str(val).strip() for val in row if pd.notna(val) and str(val).strip() != '']), 
        axis=1
    ).replace('', np.nan)
    
    # drop old cols and return the reduced df
    df = df.drop(columns = valid_cols)
    df.insert(insert_loc, "Full Address", temp_full_address)
    
    return df

# Normalize Gender Column
def gender_proc(df):

    # check required cols exist
    if "Gender" not in df.columns:
        print("No Gender columns found")
        return df
    
    def clean_gender(value):
        if pd.isna(value):
            return None
        
        val_str = str(value).strip()
        
        if val_str.lower() == 'male':
            return 'Male'
        elif val_str.lower() == 'female':
            return 'Female'
        else:
            return None

    df['Gender'] = df['Gender'].apply(clean_gender)

    return df

# Join Columns Into One Place of Birth
def pob_proc(df):
    cols = ["Town of birth", "Country of birth"]
    
    valid_cols = [c for c in cols if c in df.columns]
    if not valid_cols:
        print("Check Columns")
        return df

    insert_loc = df.columns.get_loc("Town of birth")
    
    def combine_pob(row):
        parts = [str(row[c]).strip() for c in valid_cols if pd.notna(row[c]) and str(row[c]).strip() != '']
        # Join with comma and space
        return ", ".join(parts) if parts else np.nan

    temp_pob = df[valid_cols].apply(combine_pob, axis=1)
    
    # 4. Drop the old columns and insert the new one
    df = df.drop(columns=valid_cols)
    df.insert(insert_loc, "Place of Birth", temp_pob)
    
    return df

def dob_proc(df):

    # check col exists
    if "D.O.B" not in df.columns:
        print("Check columns")
        return df
        
    def clean_dob(value):
        if pd.isna(value):
            return np.nan
        cleaned = str(value).strip().upper() # standardise
        
        if cleaned == '': # if string empty
                return np.nan
        return cleaned
    df["D.O.B"] = df["D.O.B"].apply(clean_dob)
    
    return df

In [8]:
# Main func - choose which to apply
def preprocess_data(df):
    df = numeric_proc(df)
    df = address_proc(df)
    df = gender_proc(df)
    df = pob_proc(df)
    df = dob_proc(df)
    return df

In [9]:
procdf = preprocess_data(df)

In [14]:
# export outputs
def export_relational_tables(df):

    # set own output directory
    output_dir = ".../UK_Sanctions_CaseStudy/output"
    os.makedirs(output_dir, exist_ok=True)
    
# entities table
    entities = df[df["Name type"] == "Primary Name"].copy()
    
    # keep only the core columns for the main entities table
    core_cols = [
        "Unique ID", "Designation Type", "First Name", "Surname", "Full Name", 
        "D.O.B", "Place of Birth", "Gender", "Nationality(/ies)", 
        "Passport number", "National Identifier number", "IMO number",
        "Sanctions Imposed", "Date Designated"
    ]
    # filter to cols that exist (in case of changes)
    entities = entities[[c for c in core_cols if c in entities.columns]]
    
    # drop exact duplicates just in case
    entities = entities.drop_duplicates(subset = ["Unique ID"])
    
# alias table
    aliases = df[df["Name type"] != "Primary Name"].copy()
    
    alias_cols = ["Unique ID", "Full Name", "Alias strength"]
    aliases = aliases[[c for c in alias_cols if c in aliases.columns]]
    
    # rename to 'Alias Name' for clarity
    aliases = aliases.rename(columns={"Full Name": "Alias Name"})
    
    # drop rows where alias name is blank/drop exact duplicates
    aliases = aliases.dropna(subset=["Alias Name"]).drop_duplicates()
    
# address table
    addresses = df[["Unique ID", "Full Address", "Address Country", "Phone number", "Email address", "Website"]].copy()
    # include phone number/email address etc. in case it's different for each entities' address
    # drop rows where the address NaN/duuplicates
    addresses = addresses.dropna(subset=["Full Address"]).drop_duplicates()
    
# export to csv
    df.to_csv(f"{output_dir}/master_df.csv", index=False) # main table, no dropped cols
    entities.to_csv(f"{output_dir}/entities.csv", index=False)
    aliases.to_csv(f"{output_dir}/aliases.csv", index=False)
    addresses.to_csv(f"{output_dir}/addresses.csv", index=False)
    
    print(f"Created Tables: Entities ({len(entities)} rows), Aliases ({len(aliases)} rows), and Addresses ({len(addresses)} rows).")

In [11]:
export_relational_tables(procdf)

Created Tables: Entities (3680 rows), Aliases (11369 rows), and Addresses (5580 rows).
